In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import plotly.express as px

df = pd.read_csv('../data/raw/clinical_trials_data.csv')
print(f"Loaded {len(df)} trials")

Loaded 2000 trials


In [2]:
# Generate abstract text for each trial
disease_keywords = {
    'Oncology': ['tumor','cancer','chemotherapy','immunotherapy','EGFR','PD-L1','metastatic','radiation'],
    'Cardiovascular': ['cardiac','heart failure','arrhythmia','statin','anticoagulant','coronary','hypertension'],
    'Neurology': ['Alzheimer','Parkinson','seizure','neuropathy','dopamine','cognitive','dementia'],
    'Infectious Disease': ['bacterial','viral','antibiotic','vaccine','pathogen','HIV','antiviral','sepsis'],
    'Diabetes/Metabolic': ['insulin','glucose','HbA1c','obesity','GLP-1','pancreatic','hyperglycemia'],
    'Respiratory': ['asthma','COPD','pulmonary','bronchial','FEV1','inhaler','fibrosis'],
    'Immunology/Autoimmune': ['autoimmune','TNF','cytokine','inflammation','biologics','JAK','rheumatoid'],
    'Rare Disease': ['orphan','enzyme replacement','genetic mutation','rare variant','biomarker','monogenic'],
    'Mental Health': ['depression','anxiety','SSRI','schizophrenia','bipolar','neurotransmitter'],
    'Other': ['clinical','patient','treatment','efficacy','safety','dose','randomized']
}

def generate_abstract(row):
    kws = disease_keywords.get(row['disease_area'], disease_keywords['Other'])
    selected = np.random.choice(kws, size=min(8, len(kws)), replace=True)
    if row['ai_designed']:
        selected = np.append(selected, ['machine learning','AI','computational','in silico'])
    if row['has_biomarker']:
        selected = np.append(selected, ['biomarker','genomic','precision medicine'])
    return ' '.join(selected)

np.random.seed(42)
df['abstract_text'] = df.apply(generate_abstract, axis=1)
print("Sample abstracts:")
for _, row in df.head(3).iterrows():
    print(f"  [{row['disease_area']}] {row['abstract_text']}")

Sample abstracts:
  [Other] randomized efficacy safety randomized treatment safety safety biomarker genomic precision medicine
  [Diabetes/Metabolic] hyperglycemia glucose HbA1c hyperglycemia HbA1c HbA1c GLP-1 biomarker genomic precision medicine
  [Diabetes/Metabolic] obesity HbA1c pancreatic GLP-1 glucose obesity pancreatic


In [3]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=200, ngram_range=(1, 2))
X = tfidf.fit_transform(df['abstract_text'])

le = LabelEncoder()
y = le.fit_transform(df['disease_area'])

print(f"Feature matrix shape: {X.shape}")
print(f"Classes: {list(le.classes_)}")

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Feature matrix shape: (2000, 200)
Classes: ['Cardiovascular', 'Diabetes/Metabolic', 'Immunology/Autoimmune', 'Infectious Disease', 'Mental Health', 'Neurology', 'Oncology', 'Other', 'Rare Disease', 'Respiratory']

Accuracy: 1.000

Classification Report:
                       precision    recall  f1-score   support

       Cardiovascular       1.00      1.00      1.00        57
   Diabetes/Metabolic       1.00      1.00      1.00        31
Immunology/Autoimmune       1.00      1.00      1.00        37
   Infectious Disease       1.00      1.00      1.00        48
        Mental Health       1.00      1.00      1.00        13
            Neurology       1.00      1.00      1.00        39
             Oncology       1.00      1.00      1.00       116
                Other       1.00      1.00      1.00        14
         Rare Disease       1.00      1.00      1.00        16
          Respiratory       1.00      1.00      1.00        29

             accuracy                           1.0

In [4]:
# Top TF-IDF terms per disease area
feature_names = tfidf.get_feature_names_out()
top_terms = {}

for i, disease in enumerate(le.classes_):
    indices = np.where(y == i)[0]
    mean_tfidf = np.array(X[indices].mean(axis=0)).flatten()
    top_idx = mean_tfidf.argsort()[-8:][::-1]
    top_terms[disease] = [feature_names[j] for j in top_idx]

# Display as dataframe
terms_df = pd.DataFrame(top_terms).T
terms_df.columns = [f'Term_{i+1}' for i in range(8)]
print("Top TF-IDF terms per disease area:")
print(terms_df)

import json
with open('../data/processed/top_terms.json', 'w') as f:
    json.dump(top_terms, f, indent=2)
print("\nSaved to data/processed/top_terms.json")

Top TF-IDF terms per disease area:
                              Term_1         Term_2         Term_3  \
Cardiovascular            arrhythmia  anticoagulant       coronary   
Diabetes/Metabolic     hyperglycemia        glucose            glp   
Immunology/Autoimmune            tnf            jak   inflammation   
Infectious Disease        antibiotic        vaccine       pathogen   
Mental Health                bipolar     depression  schizophrenia   
Neurology                  parkinson     neuropathy      alzheimer   
Oncology                       tumor      radiation         cancer   
Other                     randomized         safety       clinical   
Rare Disease                  orphan      monogenic        variant   
Respiratory                  inhaler      bronchial         asthma   

                             Term_4      Term_5            Term_6  \
Cardiovascular         hypertension     cardiac            statin   
Diabetes/Metabolic            hba1c     insulin        p